# Solving the Puzzle of the Wolf, the Goat, and the Cabbage

This notebook accompanies the section *Reachable States* of the lecture notes.  Several algorithms for finite state
machines need to know which states can be reached from the start state.  In this notebook we introduce the function
`reachable`, which checks whether a goal state can be reached from a start state and computes a path that leads to
it.  As an application, we solve a classic river crossing puzzle:  A farmer has to take a wolf, a goat, and a cabbage
across a river.  We describe the crossings of the river as a binary relation on the set of states of the puzzle.
Then a solution of the puzzle is a path from the start state to the goal state in the graph of this relation.  The
function `reachable` computes such a path with a *fixed-point iteration*.  The notebook
`ReflexiveTransitiveClosure.ipynb` solves the puzzle of the missionaries and the infidels with the same technique.

**Contents**

1. [The Puzzle](#The-Puzzle)
2. [Modelling the States](#Modelling-the-States)
   - [The Function `power_set`](#The-Function-power_set)
   - [The Function `problem`](#The-Function-problem)
3. [Modelling the Crossings as a Binary Relation](#Modelling-the-Crossings-as-a-Binary-Relation)
4. [Finding a Path with a Fixed-Point Iteration](#Finding-a-Path-with-a-Fixed-Point-Iteration)
   - [The Function `fixpoint`](#The-Function-fixpoint)
   - [The Function `path_product`](#The-Function-path_product)
   - [The Function `reachable`](#The-Function-reachable)
5. [Solving the Puzzle](#Solving-the-Puzzle)
6. [Displaying the Solution](#Displaying-the-Solution)
   - [The Function `relation2dot`](#The-Function-relation2dot)
   - [The Function `print_path`](#The-Function-print_path)

## The Puzzle

A farmer has to take a wolf, a goat, and a cabbage from the left bank of a river to its right bank.  There is a boat,
but it is so small that the farmer can take at most one of the three along on every crossing.  Furthermore, only the
farmer can row the boat.  However, the wolf and the goat must never be left alone on one bank of the river, since
the wolf would eat the goat.  Likewise, the goat would eat the cabbage if these two were left alone.  How can the
farmer get the wolf, the goat, and the cabbage safely to the right bank?

## Modelling the States

We call the farmer, the wolf, the goat, and the cabbage the *objects* of the puzzle.  A state of the puzzle is
represented as the set $S$ of the objects that are on the left bank of the river.  The objects that are not
elements of $S$ are on the right bank.  As the states are stored in sets, they have to be *hashable*.  Therefore, a
state is represented as a `frozenset` and not as a `set`.

In [ ]:
type State = frozenset[str]

The tuple `Objects` lists the objects in the order in which they are displayed.  The set `All` contains all
objects.  Then a state $S$ has the set $\texttt{All} \backslash S$ of the objects on the right bank as its
counterpart.  In the start state everything is on the left bank, so the start state is `All`.  The goal state is the
empty set, as everything, including the farmer, has to get to the right bank.

In [ ]:
Objects = ('farmer', 'wolf', 'goat', 'cabbage')
All: State = frozenset(Objects)

In order to display a state, we list the objects on the left bank and the objects on the right bank, separated by
a vertical bar that stands for the river.

**Function `names(S)`**
- *Input:* `S` is a set of objects.
- *Output:* A string that lists the objects in `S` in the order given by `Objects`.

In [ ]:
def names(S: State) -> str:
    return ' '.join(x for x in Objects if x in S)

**Function `state_name(S)`**
- *Input:* `S` is a state, i.e. the set of the objects on the left bank.
- *Output:* A string that lists the objects on the left bank, a vertical bar, and the objects on the right bank.

In [ ]:
def state_name(S: State) -> str:
    return names(S) + ' | ' + names(All - S)

In [ ]:
state_name(frozenset({ 'wolf', 'cabbage' }))

### The Function `power_set`

The states are subsets of the set `All`.  The set of all subsets of a set $M$ is called the *power set* of $M$ and is
written as $2^M$.  *Python* has no built-in function that computes the power set.  The function
`combinations` from the module `itertools` computes all combinations of $k$ elements of $M$, i.e. all subsets of
$M$ that have exactly $k$ elements, as tuples.  The power set of $M$ is the union of these sets for
$k = 0, 1, \cdots, |M|$.

In [ ]:
from itertools import combinations

**Function `power_set(M)`**
- *Input:* `M` is a finite set.
- *Output:* The set $2^M$ of all subsets of `M`.  As the subsets are elements of a set, they are represented as
  `frozenset`s.

In [ ]:
def power_set[T](M: frozenset[T]) -> set[frozenset[T]]:
    return { frozenset(C) for k in range(len(M) + 1)
                          for C in combinations(M, k)
           }

As `All` has $4$ elements, there are $2^4 = 16$ subsets of `All`.

In [ ]:
len(power_set(All))

### The Function `problem`

Not all of these $16$ subsets are allowed as states:  If the farmer is not on one bank of the river, then the goat
must not be on this bank together with the cabbage or together with the wolf.

**Function `problem(S)`**
- *Input:* `S` is the set of the objects on one bank of the river.
- *Output:* `True` if the farmer is not on this bank and the goat can eat the cabbage or the wolf can eat the goat,
  `False` otherwise.

In [ ]:
def problem(S: State) -> bool:
    return 'farmer' not in S and 'goat' in S and len(S) > 1

The set `States` contains all states $S$ such that there is no problem on either bank of the river, i.e. neither on
the left bank, where the objects in $S$ are, nor on the right bank, where the objects in $\texttt{All} \backslash S$
are.  The other states are not needed, as the puzzle must never reach them.

In [ ]:
States: set[State] = { S for S in power_set(All) if not problem(S) and not problem(All - S) }

Six of the $16$ subsets are excluded, so only $10$ states are left.

In [ ]:
sorted(state_name(S) for S in States)

## Modelling the Crossings as a Binary Relation

Next, we define a binary relation `Crossings` on the set `States`.  A pair $\langle S_1, S_2 \rangle$ is an element
of `Crossings` if a single crossing of the river leads from the state $S_1$ to the state $S_2$.  We represent a binary
relation as a set of pairs.

In [ ]:
type Relation = set[tuple[State, State]]

If the farmer is on the left bank in the state $S$, then the farmer takes a set $B$ of objects to the right bank.
This set $B$ has to satisfy the following conditions:
- $B \subseteq S$, as only objects on the left bank can be taken to the right bank.
- $\texttt{farmer} \in B$, as only the farmer can row the boat.
- $|B| \leq 2$, as the farmer can take at most one more object along.
- The new state $S \backslash B$ has to be an element of `States`, i.e. there must not be a problem on either bank
  after the crossing.

Hence, the crossings from the left bank to the right bank are described by the relation
$$
  \texttt{LeftToRight} := \bigl\{ \langle S, S \backslash B \rangle \bigm|
  S \in \texttt{States} \wedge B \in 2^S \wedge \texttt{farmer} \in B \wedge |B| \leq 2 \wedge
  S \backslash B \in \texttt{States} \bigr\}.
$$

In [ ]:
LeftToRight: Relation = { (S, S - B) for S in States
                                     for B in power_set(S)
                                     if  'farmer' in B and len(B) <= 2 and S - B in States
                        }

Every crossing from the right bank to the left bank undoes a crossing from the left bank to the right bank:  If the
farmer takes the objects in $B$ from the right bank to the left bank, this leads from a state $S \backslash B$ to the
state $S$.  Hence, the crossings from the right bank to the left bank are described by the *inverse* of the relation
`LeftToRight`, and the relation `Crossings` is the union of `LeftToRight` and its inverse.

In [ ]:
RightToLeft: Relation = { (S2, S1) for (S1, S2) in LeftToRight }

In [ ]:
Crossings = LeftToRight | RightToLeft
len(Crossings)

## Finding a Path with a Fixed-Point Iteration

If we interpret the relation `Crossings` as a directed graph, a solution of the puzzle is a *path* from the start
state to the goal state in this graph, i.e. a list $[x_0, x_1, \cdots, x_n]$ of states such that $x_0$ is the start
state, $x_n$ is the goal state, and $\langle x_{i-1}, x_i \rangle \in \texttt{Crossings}$ holds for all
$i \in \{1, \cdots, n\}$.  The number $n$ is the *length* of the path, i.e. the number of crossings.  In *Python*,
we represent a path as a tuple of states.

In [ ]:
type Path = tuple[State, ...]

In order to find a path from the start state to the goal state, we compute *sets of paths* with a fixed-point
iteration.  It starts with the set that contains only the path $(x_0)$ of length $0$, which consists of the start
state $x_0$ alone.  In every step, all paths found so far are extended by one more crossing.

### The Function `fixpoint`

The function `fixpoint` implements the fixed-point iteration.  It is the same function as in the notebook
`01-NFA-2-DFA.ipynb`, where its type annotation is explained:  Given a function $f$ and a start value $x$, it
computes the values
$$ x,\; f(x),\; f\bigl(f(x)\bigr),\; f\Bigl(f\bigl(f(x)\bigr)\Bigr),\; \cdots $$
until it finds a value $y$ such that $f(y) = y$, i.e. a *fixed point* of $f$.  The additional arguments `args` are
passed to `f` in every call.  The function `reachable`, which is defined below, uses them to pass the goal state and
the relation to its step function.

**Function `fixpoint(f, x, *args)`**
- *Input:* `f` is a function, `x` is the start value of the iteration, and `args` are the additional arguments of `f`.
- *Output:* The first value `y` of the sequence `x`, `f(x, *args)`, `f(f(x, *args), *args)`, $\cdots$ such that `f(y, *args) == y`.

In [ ]:
from collections.abc import Callable

In [ ]:
def fixpoint[T, *Ts](f: Callable[[T, *Ts], T], x: T, *args: *Ts) -> T:
    while True:
        y = f(x, *args)
        if y == x:
            return x
        x = y

### The Function `path_product`

A single step of the iteration extends every path $(x_0, \cdots, x_n)$ in a set $P$ of paths by every state $y$ such
that $\langle x_n, y \rangle \in R$.  The result is called the *path product* $P \bullet R$ of $P$ and $R$:
$$ P \bullet R := \bigl\{ (x_0, \cdots, x_n, y) \bigm| (x_0, \cdots, x_n) \in P \wedge \langle x_n, y \rangle \in R
   \wedge y \not\in \{ x_0, \cdots, x_n \} \bigr\}.
$$
It is the counterpart of the composition $R_1 \circ R_2$ of two relations in the notebook
`ReflexiveTransitiveClosure.ipynb`, which only keeps the first and the last element of a path.  The path product
leaves out the paths that visit a state twice:  If a path visits a state twice, the part of the path between the
two visits is a detour that can be cut out.  As there are only finitely many paths that do not visit a state twice,
this guarantees that the iteration terminates.

**Function `path_product(P, R)`**
- *Input:* `P` is a set of paths and `R` is a binary relation.
- *Output:* The set of all paths that result from extending a path in `P` by a single state that can be reached from
  its last state with `R` and that does not already occur in this path.

In [ ]:
def path_product(P: set[Path], R: Relation) -> set[Path]:
    return { p + (y,) for p in P
                      for (x, y) in R
                      if  p[-1] == x and y not in p
           }

### The Function `reachable`

The function `reachable(start, goal, R)` checks whether the state `goal` can be reached from the state `start` with
the relation `R` and, if so, it returns a path from `start` to `goal`.  It works as follows:
- The fixed-point iteration starts with the set $\{ (\texttt{start}) \}$.
- The function `reachable_step` computes a single step of the iteration:  It adds the path product
  $\texttt{Paths} \bullet R$ to the set `Paths` of the paths that have been found so far.  After $n$ steps, `Paths`
  contains all paths of length at most $n$ that start in `start` and do not visit a state twice.
- As soon as `Paths` contains a path that ends in `goal`, the function `reachable_step` returns the set `Paths`
  unchanged.  Hence `Paths` is a fixed point of `reachable_step` and the iteration stops.  At this point, the paths
  in `Paths` that end in `goal` are the shortest paths from `start` to `goal`.
- If `goal` can not be reached from `start`, the iteration stops as soon as no new paths are found.  Then there is
  no path that ends in `goal` and `reachable` returns the empty tuple `()`.  The empty tuple is not a path, since
  every path contains at least the state it starts with.

There can be several shortest paths from `start` to `goal`.  As the order in which the elements of a set are
enumerated is not specified, `reachable` picks the path whose list of state names is the smallest with respect to
the lexicographic order.  This way, the result does not change when the notebook is executed again.

The notebooks `08-Minimize.ipynb` and `08-Minimize-Table.ipynb` use a simpler variant of `reachable`.  It only
computes the *set* of all states of a <span style="font-variant:small-caps;">Dfa</span> that can be reached from its
start state, while the function `reachable` defined here also computes a *path* that leads to the goal state.  The
function `findPaths` in the notebook `ReflexiveTransitiveClosure.ipynb` uses the same algorithm as the function
`reachable` defined here, but it returns all shortest paths.

In the worst case, the number of paths grows exponentially with the number of states.  For big graphs, the search
algorithms presented in the lecture on artificial intelligence, e.g. *breadth-first search*, are more efficient.
For our puzzle, the number of paths is small.

**Function `reachable_step(Paths, goal, R)`**
- *Input:* `Paths` is a set of paths, `goal` is a state, and `R` is a binary relation on states.
- *Output:* If a path in `Paths` ends in `goal`, the set `Paths` itself.  Otherwise, the set `Paths` together with
  the path product of `Paths` and `R`.

In [ ]:
def reachable_step(Paths: set[Path], goal: State, R: Relation) -> set[Path]:
    if { p for p in Paths if p[-1] == goal } != set():
        return Paths
    return Paths | path_product(Paths, R)

**Function `path_name(p)`**
- *Input:* `p` is a path.
- *Output:* The list of the names of the states of `p`.  It is used to compare paths.

In [ ]:
def path_name(p: Path) -> list[str]:
    return [state_name(S) for S in p]

**Function `reachable(start, goal, R)`**
- *Input:* `start` and `goal` are states and `R` is a binary relation on states.
- *Output:* A shortest path from `start` to `goal`, if `goal` can be reached from `start`, and the empty tuple
  otherwise.

In [ ]:
def reachable(start: State, goal: State, R: Relation) -> Path:
    Paths = fixpoint(reachable_step, { (start,) }, goal, R)
    Found = { p for p in Paths if p[-1] == goal }
    if Found == set():
        return ()
    return min(Found, key=path_name)

## Solving the Puzzle

In the start state, everything is on the left bank.  In the goal state, everything is on the right bank, so the
left bank is empty.

In [ ]:
start: State = All
goal:  State = frozenset()

In [ ]:
solution = reachable(start, goal, Crossings)

If the puzzle had no solution, `reachable` would have returned the empty tuple.  Hence the puzzle is solvable.

In [ ]:
solution != ()

The solution needs $7$ crossings.

In [ ]:
len(solution) - 1

In [ ]:
[state_name(S) for S in solution]

## Displaying the Solution

### The Function `relation2dot`

In order to visualize the relation `Crossings` and the solution, we use the package
[`graphviz`](https://graphviz.readthedocs.io).

In [ ]:
import graphviz as gv

The function `relation2dot(M, R, p)` is a variant of the function with the same name in the notebook
`ReflexiveTransitiveClosure.ipynb`.  It turns every state in $M$ into a node and every pair
$\langle S_1, S_2 \rangle \in R$ into an edge.  As the relation `Crossings` is symmetric, the two edges
$\langle S_1, S_2 \rangle$ and $\langle S_2, S_1 \rangle$ are drawn as a single edge with arrow heads at both ends.
The set `Drawn` contains the pairs that have already been drawn.  The nodes and edges of the path `p` are drawn in
red.

The graph is drawn with the layout engine `neato`, which uses a *spring model*:  Every edge pulls its two nodes
together like a spring.  The attribute `len` specifies the preferred length of the edges.  As the orientation of a
drawing produced by `neato` is arbitrary, the attribute `normalize` rotates the drawing such that the first edge
points from left to right.  The edges of the path `p` are drawn first, so the start state is on the left.  The other
nodes and edges are sorted according to their names.  This way, the layout of the graph does not change when the
notebook is executed again.

**Function `edge_name(e)`**
- *Input:* `e` is a pair of states.
- *Output:* The pair of the names of these states.  It is used to sort the edges.

In [ ]:
def edge_name(e: tuple[State, State]) -> tuple[str, str]:
    return state_name(e[0]), state_name(e[1])

**Function `relation2dot(M, R, p)`**
- *Input:* `M` is a set of states, `R` is a symmetric binary relation on `M`, and `p` is a path.
- *Output:* A `graphviz` graph of `R`.  The nodes and edges of `p` are drawn in red.

In [ ]:
def relation2dot(M: set[State], R: Relation, p: Path) -> gv.Digraph:
    PathEdges = [ (p[i], p[i+1]) for i in range(len(p) - 1) ]
    dot = gv.Digraph(engine='neato')
    dot.graph_attr['normalize'] = '0'
    dot.node_attr['shape']      = 'box'
    dot.edge_attr['len']        = '3.2'
    dot.edge_attr['dir']        = 'both'
    for S in sorted(M, key=state_name):
        if S in p:
            dot.node(state_name(S), color='red', fontcolor='red')
        else:
            dot.node(state_name(S))
    Drawn: Relation = set()
    for (S1, S2) in PathEdges:
        dot.edge(state_name(S1), state_name(S2), color='red', penwidth='2')
        Drawn.add((S1, S2))
    for (S1, S2) in sorted(R, key=edge_name):
        if (S1, S2) in Drawn or (S2, S1) in Drawn:
            continue
        dot.edge(state_name(S1), state_name(S2))
        Drawn.add((S1, S2))
    return dot

The graph shows that the puzzle has two solutions:  After the farmer has taken the goat to the right bank and
returned, the farmer can either take the wolf or the cabbage across next.  The function `reachable` has picked the
solution in which the farmer takes the wolf across first, since the name `'cabbage | farmer wolf goat'` of the state
that is reached by this crossing is smaller than the name `'wolf | farmer goat cabbage'` of the state that is reached
by taking the cabbage across.

In [ ]:
relation2dot(States, Crossings, solution)

### The Function `print_path`

Finally, we print the solution in a more readable form.  Every state is printed as a line that shows the objects on
the left bank, the river, and the objects on the right bank.  Every object has a column of its own on either bank,
so that the objects stay in place when other objects cross the river.  Between two states, a line shows which
objects cross the river in which direction.

**Function `bank(S)`**
- *Input:* `S` is the set of the objects on one bank of the river.
- *Output:* A string that contains the names of the objects in `S`.  The objects that are not in `S` are replaced
  by blanks.

In [ ]:
def bank(S: State) -> str:
    return ' '.join(x if x in S else ' ' * len(x) for x in Objects)

The string `River` is printed between the two banks.  The number `Width` is the length of the lines that display the
states.  It is used to center the lines that display the crossings.

In [ ]:
River = '   |~~~~~|   '
Width = 2 * len(bank(All)) + len(River)

**Function `print_state(S)`**
- *Input:* `S` is a state.
- *Output:* None.  The function prints the objects on the left bank, the river, and the objects on the right bank.

In [ ]:
def print_state(S: State) -> None:
    print(bank(S) + River + bank(All - S))

**Function `print_boat(S1, S2)`**
- *Input:* `S1` and `S2` are states such that a single crossing leads from `S1` to `S2`.
- *Output:* None.  The function prints the objects in the boat and the direction of the crossing.  If the farmer is
  on the left bank in the state `S1`, the boat crosses from left to right and carries the objects in
  $S_1 \backslash S_2$.  Otherwise, it crosses from right to left and carries the objects in $S_2 \backslash S_1$.

In [ ]:
def print_boat(S1: State, S2: State) -> None:
    if 'farmer' in S1:
        print(f'>>>> {names(S1 - S2)} >>>>'.center(Width))
    else:
        print(f'<<<< {names(S2 - S1)} <<<<'.center(Width))

**Function `print_path(p)`**
- *Input:* `p` is a path, i.e. a non-empty tuple of states such that a single crossing leads from every state to
  the next state.
- *Output:* None.  The function prints all states of the path and the crossings between them.

In [ ]:
def print_path(p: Path) -> None:
    for i in range(len(p) - 1):
        print_state(p[i])
        print_boat(p[i], p[i+1])
    print_state(p[-1])

In [ ]:
print_path(solution)